# 00 · Getting the data

Run this notebook top to bottom once. It gets the data onto your machine and shows how to use it.
After that, any notebook can load the data with three lines (section 4).

## 1. Our goal

**Does the text of a company's annual report predict how volatile its share price will be over
the next year, beyond what last year's volatility already tells us?**

Last year's volatility is already a strong predictor, so that is our baseline. We build neural
networks that read the report and see whether they beat it. The full plan is in
[`Oscar/PLAN.md`](../Oscar/PLAN.md).

## 2. What the data is

**27,142 annual reports** from US public companies, filed 1996–2006. For each report we have:

- **Two numbers about the share price:**
  - `logvol_before`: volatility over the 12 months before the report. **The baseline input.**
  - `logvol_after`: volatility over the 12 months after. **What we predict.**

  Both are logs, which is why they're negative (around −3.4 typically).
- **Two sections of the report's text, already turned into numbers:**
  - **Item 1**, the business description: what the company does, its markets and competitors.
  - **Item 7**, management's discussion of the year.

  Each section is cut into **chunks** of about 400 words, and each chunk has been turned into a
  list of **768 numbers** that captures what it says. A typical section is 14–16 chunks, at most 64.
  So each report's text is a short sequence of 768-number vectors per section.

## 3. How to get it

**Step 1: add the shared OneDrive folder.** Oscar has shared a OneDrive folder called **`sprint2-data`**
with you (check your Bristol email or OneDrive's *Shared* page). Open it in OneDrive on the web and
click **Add shortcut to My files**. It then syncs to your computer like your other OneDrive files.
It's about 1.4 GB. Right-click the folder and choose **Always keep on this device** so the copy
below doesn't stall. Don't edit the files in it: it's everyone's copy.

**Step 2: set up Python**, if you haven't already. From the repo root:
`python3 -m venv .venv`, `source .venv/bin/activate`, `pip install -r requirements.txt`.

**Step 3: run the two cells below.** The first finds the folder. If it can't, paste the folder's path
into `DATA_DIR` (on a Mac: right-click the folder in Finder, hold Option, *Copy as Pathname*).
The second copies the files into `data/processed/` in your copy of the repo, which is where our
code looks. Never commit them: `data/processed/` is in `.gitignore` already.

In [ ]:
import shutil
import sys
from pathlib import Path

DATA_DIR = None   # e.g. Path("/Users/you/OneDrive - University of Bristol/sprint2-data") if not found

FILES = ["dataset.csv", "item1_emb.npy", "item7_emb.npy"]

# The repo root is the folder above this notebook's folder.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "Oscar"))   # so we can use Oscar/build_dataset.py

def find_data():
    home = Path.home()
    onedrives = [home / "OneDrive - University of Bristol",
                 home / "Library" / "CloudStorage" / "OneDrive-UniversityofBristol"]
    for od in onedrives:
        for candidate in [od / "sprint2-data", od / "DST" / "sprint2-data"]:
            if (candidate / "dataset.csv").exists():
                return candidate
    return None

src = Path(DATA_DIR) if DATA_DIR else find_data()
assert src is not None, "Folder not found: set DATA_DIR above."
missing = [f for f in FILES if not (src / f).exists()]
assert not missing, f"The shared folder is missing {missing}: ask Oscar."
print(f"Found the shared folder: {src}")

In [ ]:
processed = ROOT / "data" / "processed"
processed.mkdir(parents=True, exist_ok=True)

for f in FILES:
    shutil.copy(src / f, processed / f)
    print(f"copied {f:15s} {(processed / f).stat().st_size / 1e6:7.1f} MB")

You now have three files:

| File | Contents |
|---|---|
| `dataset.csv` | one row per report |
| `item1_emb.npy` | every Item 1 chunk, stacked into one table: 444,394 rows × 768 |
| `item7_emb.npy` | the same for Item 7: 498,607 rows × 768 |

The columns of `dataset.csv` you'll use:

| Column | Meaning |
|---|---|
| `year`, `date` | year and date of the report |
| `name` | company name |
| `logvol_before` | volatility, 12 months before (**baseline input**) |
| `logvol_after` | volatility, 12 months after (**what we predict**) |
| `item1_n`, `item7_n` | number of chunks this report has |
| `item1_start`, `item7_start` | the row where this report's chunks start in the `.npy` table |

Reports have different numbers of chunks, so the chunks are stacked into one long table and the
`start` and `n` columns say which rows belong to which report. The `chunks()` helper below does
this for you.

**Check it worked:**

In [ ]:
from build_dataset import load_dataset, chunks

df, emb = load_dataset()
print(f"{len(df):,} reports, years {df.year.min()}-{df.year.max()}")
print(f"Item 1 chunks: {emb['item1'].shape},  Item 7 chunks: {emb['item7'].shape}")
assert len(df) == 27142 and emb["item7"].shape == (498607, 768), "Something is missing: ask Oscar."
df.head()

## 4. What we do with it

In your own notebooks, load the data with:

```python
import sys; sys.path.insert(0, "../Oscar")
from build_dataset import load_dataset, chunks
df, emb = load_dataset()
```

There are two ways to feed a report to a model.

**One report as a sequence of chunks.** This is the input for our attention network, which learns
which chunks matter:

In [ ]:
i = 0
seq = chunks(df, emb, i, "item7")
print(df.loc[i, "name"], df.loc[i, "year"], "| Item 7 as", seq.shape, "(chunks x 768)")

**Every report as one vector.** Averaging each report's chunks gives a plain
(reports × 768) table that scikit-learn or a simple neural network can use directly:

In [ ]:
import numpy as np

def mean_pool(section):
    sums = np.add.reduceat(emb[section], df[f"{section}_start"].to_numpy(), axis=0, dtype=np.float32)
    return sums / df[f"{section}_n"].to_numpy()[:, None]

X7 = mean_pool("item7")
print(X7.shape)

**Training and testing.** We always train on earlier years and test on a later one, so a model
never sees the future: for example, train on 2001–2005 and test on 2006. Models are scored by mean
squared error (MSE) on `logvol_after`. The number to beat is the baseline that predicts next year's
volatility is the same as last year's:

In [ ]:
train = df.year.between(2001, 2005).to_numpy()
test = (df.year == 2006).to_numpy()

mse = np.mean((df.logvol_after[test] - df.logvol_before[test]) ** 2)
print(f"train {train.sum():,} reports, test {test.sum():,} reports")
print(f"Baseline 'same as last year' MSE on 2006: {mse:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(df.logvol_before, df.logvol_after, s=2, alpha=0.2)
lims = [-8, 0]
ax.plot(lims, lims, color="k", lw=1)
ax.set(xlim=lims, ylim=lims, xlabel="log volatility, year before",
       ylabel="log volatility, year after", title="Last year's volatility is a strong baseline")
plt.show()